# Detect spots in 3D from Napari

- We must start by importing what we need since we don't want to re-implement stuff by ourselves:
    - `skimage.morphology` contains function to perform the **opening by reconstruction** and the search of **local maxima**.
    - `numpy` allows to store and perform operations on N-dimensional arrays of data (like images).

In [ ]:
from skimage import morphology as morph
import numpy as np

- At this point, we can capture from Napari's viewer the layers containing the spots and the segmented nuclei.
- This code will run in the current context of Napari so some global objects already exist like `viewer`, which is the current viewer.
- We can access its layers list to grab the two layers we are interested in.
- You have to change `name of the spots layer` and `name of the SEGMENTED nuclei layer` to match the actual names of layers. 

In [ ]:
spots_layer  = viewer.layers['name of the spots layer']
nuclei_layer = viewer.layers['name of the SEGMENTED nuclei layer']

- Layers are Napari layers, we are interested in the pixel values stored inside.
- We can access them through the `.data` attribute.

In [ ]:
im_spots = spots_layer.data
im_nuclei = nuclei_layer.data

- As said in the exercise, we arbitrarily decide that for a maxima to be a spot, it must be 1.5 the standard deviation of the whole image.
- We won't process the standard deviation ourselves, numpy does it very well.
- You can adjust the factor if you think that it's too low/high.
- Our desired prominence is then stored in the variable `h`.

In [ ]:
h = np.std(im_spots) * 1.5

- We can now remove the potential background non-uniformity and the big structures using the opening by reconstruction.

In [ ]:
im_recon = morph.reconstruction(im_spots - h, im_spots)

- Once it is done, we will search for local maximas on the resulting image using the `h_maxima` function.
- This function produce a new image having the size of the original image where all pixels are at 0 except the ones located on a local maxima.
- This is a waste of space, so we are going to use `np.where` to locate the 3D coordinates of pixels that are higher than 0.
- Eventually, we will read in the segmented nuclei image the label corresponding to the coordinates of each spot.

In [ ]:
bw = morph.h_maxima(im_spots, h)
maximas = np.where(bw > 0)
points = np.column_stack(maximas)
nucleus_indices = im_nuclei[maximas]

- For visualization and control purposes, we would like to be able to see the spots that we found.
- We can do that by addind them in the current viewer as a new layer containing points.

In [ ]:
pts = viewer.add_points(
    points,
    name=f"spots_{h}",
    scale=nuclei_layer.scale,
    units=nuclei_layer.units,
    axis_labels=nuclei_layer.axis_labels,
    out_of_slice_display=True,
    size=4,
    face_color='transparent',
    border_color='white'
)

- The last step would be to count for each nucleus, how many spots it contains.
- To do so, the first thing that we will do is to compute how many nuclei we have in our image.
- We are supposed to have one label (== one value) per nucleus + 0 for the background, so knowing how many different values we have in our image would do the trick.
- Luckily, it is exactly what `np.unique` does.
- We create as many counters as we have nuclei and we initialize them at zero. To make it easier to manipulate, this counters will be stored as an array and we will say that the `i-th` item in this array is for the nucleus having the value `i`.

In [ ]:
nNuclei = len(np.unique(im_nuclei))
nSpotsPerNucleus = np.zeros((nNuclei,), dtype=np.uint8)

- Now we can simply loop on our array of labels.
- For each label, we increase the corresponding counter.
- Since 0 is supposed to be the background, we skip what we found with this value.
- To avoid losing our work, we store the result in the **features map** of the layer containing segmented nuclei.

In [ ]:
for index in nucleus_indices:
    if index == 0:
        continue
    nSpotsPerNucleus[index] += 1

# Inject the result in the features map
nuclei_layer.features['nSpots'] = nSpotsPerNucleus